# 02 — Data Preprocessing & Feature Engineering

**ECE1513 Course Project — Traffic Congestion Prediction near U of T St. George Campus**

This notebook takes the raw traffic speed-bin and weather data, applies cleaning and feature engineering, and produces the final processed dataset used by all downstream models. Steps:

1. Load raw traffic and weather data.
2. Filter to U of T area and compute average speeds from speed bins.
3. Clean data (remove zero-volume records, handle missing values).
4. Merge traffic and weather data.
5. Engineer temporal and contextual features.
6. Create congestion labels appropriate for city streets.
7. Show distributions and summary statistics.
8. Temporal train/test split (80/20) and save to `data/processed/`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

from src.data_loader import get_holidays

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Raw Data

In [ ]:
# --- Traffic data ---
traffic_raw = pd.read_csv('../data/raw/svc_raw_data_speed_2020_2024.csv')
print(f'Raw traffic shape: {traffic_raw.shape}')

# --- Weather data ---
weather_files = sorted(glob.glob('../data/raw/weather/toronto_pearson_*.csv'))
weather_raw = pd.concat([pd.read_csv(f) for f in weather_files], ignore_index=True)
print(f'Raw weather shape: {weather_raw.shape}')

## 2. Filter to U of T Area and Compute Average Speed

We restrict to locations within ~1.5 km of U of T St. George campus and compute a weighted average speed from the speed-bin counts.

In [ ]:
# Filter to U of T bounding box
LAT_MIN, LAT_MAX = 43.645, 43.675
LON_MIN, LON_MAX = -79.415, -79.385

mask = (
    (traffic_raw['latitude'] >= LAT_MIN) & (traffic_raw['latitude'] <= LAT_MAX) &
    (traffic_raw['longitude'] >= LON_MIN) & (traffic_raw['longitude'] <= LON_MAX)
)
traffic_df = traffic_raw[mask].copy()
print(f'Filtered to U of T area: {len(traffic_df):,} rows from {len(traffic_raw):,}')
print(f'Unique locations: {traffic_df["location_name"].nunique()}')

In [ ]:
# Speed bin columns and midpoints
speed_bin_cols = [
    'vol_1_19kph', 'vol_20_25kph', 'vol_26_30kph', 'vol_31_35kph',
    'vol_36_40kph', 'vol_41_45kph', 'vol_46_50kph', 'vol_51_55kph',
    'vol_56_60kph', 'vol_61_65kph', 'vol_66_70kph', 'vol_71_75kph',
    'vol_76_80kph', 'vol_81_160kph'
]

speed_bin_midpoints = np.array([10.0, 22.5, 28.0, 33.0, 38.0, 43.0, 48.0,
                                53.0, 58.0, 63.0, 68.0, 73.0, 78.0, 120.5])

bin_values = traffic_df[speed_bin_cols].values
total_vehicles = bin_values.sum(axis=1)

with np.errstate(divide='ignore', invalid='ignore'):
    avg_speed = (bin_values * speed_bin_midpoints).sum(axis=1) / total_vehicles
    avg_speed = np.where(total_vehicles > 0, avg_speed, np.nan)

traffic_df['avg_speed'] = avg_speed
traffic_df['total_volume'] = total_vehicles

print(f'Computed avg_speed for {(~np.isnan(avg_speed)).sum():,} records')
print(f'Records with zero volume (will drop): {(total_vehicles == 0).sum():,}')

## 3. Data Cleaning

In [ ]:
initial_rows = len(traffic_df)

# Parse datetime
traffic_df['time_start'] = pd.to_datetime(traffic_df['time_start'])
traffic_df['time_end'] = pd.to_datetime(traffic_df['time_end'])

# Drop records with zero total volume (no vehicles = no meaningful speed)
traffic_df = traffic_df[traffic_df['total_volume'] > 0].copy()

# Drop records with NaN avg_speed
traffic_df = traffic_df.dropna(subset=['avg_speed'])

# Drop duplicates
traffic_df = traffic_df.drop_duplicates(subset=['location_name', 'time_start', 'direction'])

# Remove extreme outlier speeds (> 120 km/h on city streets is likely error)
traffic_df = traffic_df[traffic_df['avg_speed'] <= 120].copy()

# Sort by time
traffic_df = traffic_df.sort_values('time_start').reset_index(drop=True)

print(f'Traffic after cleaning: {len(traffic_df):,} rows (dropped {initial_rows - len(traffic_df):,})')

In [ ]:
# Clean weather data
weather_df = weather_raw.copy()
weather_df['datetime'] = pd.to_datetime(weather_df['LOCAL_DATE'])
weather_df = weather_df.dropna(subset=['datetime'])

# Rename to shorter column names
weather_rename = {
    'TEMP': 'temp_c',
    'DEW_POINT_TEMP': 'dew_point_c',
    'RELATIVE_HUMIDITY': 'rel_humidity_pct',
    'WIND_SPEED': 'wind_speed_kmh',
    'VISIBILITY': 'visibility_km',
    'STATION_PRESSURE': 'pressure_kpa',
    'PRECIP_AMOUNT': 'precip_mm',
    'WEATHER_ENG_DESC': 'weather_desc',
}
weather_df = weather_df.rename(columns=weather_rename)

# Keep only relevant columns
weather_keep = ['datetime'] + [v for v in weather_rename.values() if v in weather_df.columns]
weather_df = weather_df[weather_keep].copy()

# Coerce numeric columns and interpolate small gaps
numeric_cols = ['temp_c', 'dew_point_c', 'rel_humidity_pct', 'wind_speed_kmh',
                'visibility_km', 'pressure_kpa', 'precip_mm']
for col in numeric_cols:
    if col in weather_df.columns:
        weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')

weather_df = weather_df.sort_values('datetime').reset_index(drop=True)
weather_df[numeric_cols] = weather_df[numeric_cols].interpolate(method='linear', limit=3)

# Fill remaining NaN in precip with 0 (no precipitation)
if 'precip_mm' in weather_df.columns:
    weather_df['precip_mm'] = weather_df['precip_mm'].fillna(0.0)

print(f'Weather after cleaning: {len(weather_df):,} rows')
print(f'Weather columns: {list(weather_df.columns)}')

## 4. Merge Traffic and Weather Data

We use an asof merge to join each traffic observation with the nearest hourly weather observation (within a 2-hour tolerance).

In [ ]:
# Ensure both are sorted by their datetime columns
traffic_df = traffic_df.sort_values('time_start').reset_index(drop=True)
weather_df = weather_df.sort_values('datetime').reset_index(drop=True)

merged_df = pd.merge_asof(
    traffic_df,
    weather_df,
    left_on='time_start',
    right_on='datetime',
    direction='nearest',
    tolerance=pd.Timedelta('2h')
)

# Drop the redundant weather datetime column
if 'datetime' in merged_df.columns:
    merged_df = merged_df.drop(columns=['datetime'])

print(f'Merged dataset shape: {merged_df.shape}')
print(f'Weather columns with NaN after merge:')
weather_cols_in_merged = [c for c in weather_rename.values() if c in merged_df.columns]
for col in weather_cols_in_merged:
    n_null = merged_df[col].isna().sum()
    if n_null > 0:
        print(f'  {col}: {n_null:,} ({n_null/len(merged_df)*100:.1f}%)')

## 5. Feature Engineering

Create temporal features (hour, day of week, month, is_rush_hour, is_weekend), holiday flags, and weather-derived features.

In [ ]:
import datetime

dt = merged_df['time_start']

# --- Basic temporal features ---
merged_df['hour_of_day'] = dt.dt.hour
merged_df['day_of_week'] = dt.dt.dayofweek  # 0 = Monday
merged_df['month'] = dt.dt.month
merged_df['is_weekend'] = merged_df['day_of_week'].isin([5, 6]).astype(int)

# Rush hour: 7-9 AM or 4-7 PM
merged_df['is_rush_hour'] = merged_df['hour_of_day'].apply(
    lambda h: int(h in (7, 8) or h in (16, 17, 18))
)

# --- Holidays ---
years = sorted(dt.dt.year.dropna().unique())
all_holidays = set()
for y in years:
    all_holidays.update(get_holidays(int(y)))

merged_df['is_holiday'] = dt.dt.date.isin(all_holidays).astype(int)

# Long weekend detection
long_weekend_dates = set()
for h in all_holidays:
    long_weekend_dates.add(h)
    wd = h.weekday()
    if wd == 0:  # Monday holiday -> Sat & Sun before
        long_weekend_dates.update([h - datetime.timedelta(days=1),
                                   h - datetime.timedelta(days=2)])
    elif wd == 4:  # Friday holiday -> Sat & Sun after
        long_weekend_dates.update([h + datetime.timedelta(days=1),
                                   h + datetime.timedelta(days=2)])

merged_df['is_long_weekend'] = dt.dt.date.isin(long_weekend_dates).astype(int)

# School in session (approx Sep 1 - Jun 30)
merged_df['school_in_session'] = merged_df['month'].apply(
    lambda m: int(m >= 9 or m <= 6)
).astype(int)

# --- Weather-derived binary features ---
if 'precip_mm' in merged_df.columns:
    merged_df['is_raining'] = (merged_df['precip_mm'] > 0).astype(int)

if 'visibility_km' in merged_df.columns:
    merged_df['low_visibility'] = (merged_df['visibility_km'] < 4).astype(int)

# --- Encode location as integer ---
merged_df['location_id'] = merged_df['location_name'].astype('category').cat.codes

# --- Encode direction as integer ---
merged_df['direction_code'] = merged_df['direction'].astype('category').cat.codes

print('New feature columns added:')
new_features = ['hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour',
                'is_holiday', 'is_long_weekend', 'school_in_session',
                'is_raining', 'low_visibility', 'location_id', 'direction_code']
for f in new_features:
    if f in merged_df.columns:
        print(f'  - {f}')

merged_df.head()

## 6. Create Congestion Labels

For city streets near U of T, we define congestion levels based on average speed:

| Label | Speed Range | Description |
|-------|------------|-------------|
| gridlock | < 20 km/h | Near-standstill traffic |
| heavy | 20-35 km/h | Significant delays |
| moderate | 35-50 km/h | Some slowdown from free flow |
| free_flow | >= 50 km/h | Unimpeded traffic |

In [ ]:
bins = [-np.inf, 20, 35, 50, np.inf]
labels = ['gridlock', 'heavy', 'moderate', 'free_flow']

merged_df['congestion'] = pd.cut(
    merged_df['avg_speed'], bins=bins, labels=labels, right=False
)

# Also create a numeric congestion level for modelling
congestion_map = {'gridlock': 3, 'heavy': 2, 'moderate': 1, 'free_flow': 0}
merged_df['congestion_level'] = merged_df['congestion'].map(congestion_map)

print('Congestion label distribution:')
dist = merged_df['congestion'].value_counts()
for label in labels:
    count = dist.get(label, 0)
    pct = count / len(merged_df) * 100
    print(f'  {label:>10s}: {count:>8,} ({pct:.1f}%)')

colors_map = {'gridlock': 'red', 'heavy': 'orange', 'moderate': 'gold', 'free_flow': 'green'}
colors = [colors_map[l] for l in labels]
dist.reindex(labels).plot(kind='bar', color=colors, edgecolor='white')
plt.xlabel('Congestion Level')
plt.ylabel('Count')
plt.title('Distribution of Congestion Labels (City Street Thresholds)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Before / After Statistics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(traffic_raw.loc[mask, speed_bin_cols].sum(axis=1).replace(0, np.nan).dropna(),
             bins=50, edgecolor='white', alpha=0.8)
axes[0].set_title('Total Volume per Record — Raw Data')
axes[0].set_xlabel('Total Vehicle Count')

axes[1].hist(merged_df['avg_speed'].dropna(), bins=50, edgecolor='white', alpha=0.8, color='seagreen')
axes[1].set_title('Average Speed Distribution — After Preprocessing')
axes[1].set_xlabel('Speed (km/h)')

plt.tight_layout()
plt.show()

print(f'Raw records (U of T area): {mask.sum():>10,}')
print(f'Processed records:         {len(merged_df):>10,}')
print(f'Features:                  {merged_df.shape[1]:>10}')

In [ ]:
# Summary statistics of the processed dataset
print('--- Processed Dataset Summary ---')
print(f'Shape: {merged_df.shape}')
print(f'Time range: {merged_df["time_start"].min()} to {merged_df["time_start"].max()}')
print(f'Unique locations: {merged_df["location_name"].nunique()}')
print(f'\nAverage speed stats:')
print(merged_df['avg_speed'].describe())
print(f'\nMissing values in final dataset:')
missing_final = merged_df.isnull().sum()
missing_final = missing_final[missing_final > 0]
if len(missing_final) > 0:
    print(missing_final)
else:
    print('  None')

## 8. Temporal Train / Test Split

We split chronologically — earlier data for training, later data for testing — to avoid data leakage from future observations.

In [ ]:
merged_df = merged_df.sort_values('time_start').reset_index(drop=True)

split_idx = int(len(merged_df) * 0.8)
train_df = merged_df.iloc[:split_idx].copy()
test_df = merged_df.iloc[split_idx:].copy()

print(f'Train set: {train_df.shape}  ({train_df["time_start"].min()} to {train_df["time_start"].max()})')
print(f'Test  set: {test_df.shape}  ({test_df["time_start"].min()} to {test_df["time_start"].max()})')

## 9. Save Processed Data

In [ ]:
processed_dir = os.path.join('..', 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

train_df.to_csv(os.path.join(processed_dir, 'train.csv'), index=False)
test_df.to_csv(os.path.join(processed_dir, 'test.csv'), index=False)
merged_df.to_csv(os.path.join(processed_dir, 'full_processed.csv'), index=False)

print('Saved to data/processed/:')
for f in ['train.csv', 'test.csv', 'full_processed.csv']:
    size_mb = os.path.getsize(os.path.join(processed_dir, f)) / 1e6
    print(f'  {f:30s} {size_mb:.2f} MB')

## Summary

The preprocessing pipeline:

1. Loaded midblock speed-bin data and filtered to 104 locations within ~1.5 km of U of T St. George campus.
2. Computed weighted average speed from 14 speed bins using bin midpoints.
3. Removed zero-volume records, duplicates, and extreme outlier speeds.
4. Merged traffic data with hourly weather observations from Toronto Pearson via asof join.
5. Engineered temporal features (hour, day of week, month, rush-hour flag, weekend flag), holiday/long-weekend flags, and weather-derived binary features.
6. Created four-level congestion labels using city-street-appropriate thresholds (gridlock < 20, heavy 20-35, moderate 35-50, free_flow >= 50 km/h).
7. Applied an 80/20 temporal split to preserve the time ordering of observations.

The processed `train.csv` and `test.csv` files are ready for model training in the next notebooks.